##### Traitement de données pour EDF


In [1080]:
import pandas as pd
import numpy as np


export_df = pd.read_csv('export.csv', sep=';', dtype=str)
result_df = pd.read_csv('result.csv', sep=';', dtype=str)
nomenclature_df = pd.read_csv('Nomenclature_ICPE.csv', sep=';', dtype=str)


In [1081]:
export_df.head()

,Numéro d'établissement,Nom établissement,Adresse 1,Adresse 2,Adresse 3,Code postal,Commune,Régime en vigueur,Statut SEVESO,Date de dernière inspection,Statut IED,Numéro SIRET
0,0000000002,PITCH IMMO SNC : HEXAHUB : hangars logistiques...,Secteur de la Marquette,NaN,NaN,33240,Gauriaguet,Autorisation,Non Seveso,NaN,non,42298971500186
1,0000000004,SOCIETE FERROLAC,4 RUE PIERRE GILLES DE GENNES,ZAC DE LA VIGONNIERE,NaN,18400,Saint-Florent-sur-Cher,Autorisation,Non Seveso,2023-05-16,non,78560962900064
2,0000000005,ORLEANS METROPOLE - Parc de Loire,5 PL DU 6 JUIN 1944,ESPACE SAINT MARC,45058 ORLEANS CEDEX 1,45000,Orléans,Autorisation,Non Seveso,NaN,non,24450046800040
3,0000000010,GRANUPLAST FRANCE,754 rue de la liberté,ZI la grande borne,NaN,01480,Jassans-Riottier,Non ICPE,NaN,2024-10-23,non,88325431000021
4,0000000034,EOLIEN - TOTAL QUADRAN CE La Luçoise,Forêt de Chaniaux,Lou Chambon,NaN,48250,Luc,Autorisation,Non Seveso,NaN,non,81519422000027


In [1082]:
result_df.head()

,code_aiot,x,y,code_epsg,nom_ets,num_dep,adresse,cd_insee,cd_postal,commune,...,eolienne,industrie,ied,priorite_nationale,rubriques_autorisation,rubriques_enregistrement,rubriques_declaration,date_modification,derniere_inspection,url_fiche
0,0000000002,432849,6444403,2154,PITCH IMMO SNC : HEXAHUB : hangars logistiques...,33,Secteur de la Marquette,33183,33240,Gauriaguet,...,0,1,0,0,1450|1510|1630|4801,NaN,1436|2171|2910|2925|4110|4120|4130|4140|4150|4...,2026/05/06 00:00:00,NaN,https://www.georisques.gouv.fr/risques/install...
1,0000000004,643988,6656652,2154,SOCIETE FERROLAC,18,4 RUE PIERRE GILLES DE GENNES - ZAC DE LA VIGO...,18207,18400,Saint-Florent-sur-Cher,...,0,1,0,1,2718,2712|2713,2711,2026/02/25 00:00:00,2023/05/16 00:00:00,https://www.georisques.gouv.fr/risques/install...
2,0000000005,621725,6755411,2154,ORLEANS METROPOLE - Parc de Loire,45,5 PL DU 6 JUIN 1944 - ESPACE SAINT MARC - 4505...,45274,45000,Orléans,...,0,1,0,0,NaN,NaN,NaN,2025/09/10 00:00:00,NaN,https://www.georisques.gouv.fr/risques/install...
3,0000000010,836747,6543731,2154,GRANUPLAST FRANCE,01,754 rue de la liberté - ZI la grande borne,01194,01480,Jassans-Riottier,...,0,0,0,0,NaN,NaN,NaN,2025/08/08 00:00:00,2024/10/23 00:00:00,https://www.georisques.gouv.fr/risques/install...
4,0000000034,768626,6388530,2154,EOLIEN - TOTAL QUADRAN CE La Luçoise,48,Forêt de Chaniaux - Lou Chambon,48086,48250,Luc,...,1,0,0,0,2980,NaN,NaN,2026/02/20 00:00:00,NaN,https://www.georisques.gouv.fr/risques/install...


In [1083]:
# On drop toutes les colonnes qui ne nous intéressent pas
result_df = result_df.drop(["bovins", "porcs", "volailles", "carriere", "eolienne", "industrie", "url_fiche", "code_epsg", "ied", "priorite_nationale", "num_dep"], axis =1)


In [1084]:
# On renomme les colonnes liées à l'ICPE
result_df = result_df.rename({"rubriques_autorisation" : "ICPE_A" , "rubriques_enregistrement" : "ICPE_E", "rubriques_declaration" : "ICPE_D"}, axis = 1)
result_df.columns


Index(['code_aiot', 'x', 'y', 'nom_ets', 'adresse', 'cd_insee', 'cd_postal',
       'commune', 'code_naf', 'lib_naf', 'num_siret', 'cd_regime',
       'lib_regime', 'seveso', 'lib_seveso', 'ICPE_A', 'ICPE_E', 'ICPE_D',
       'date_modification', 'derniere_inspection'],
      dtype='object')

### Dédoublement des lignes par code ICPE

In [1085]:
result_df["ICPE_A"] = result_df["ICPE_A"].apply( lambda x: f"{x}|0")
result_df["ICPE_E"] = result_df["ICPE_E"].apply( lambda x: f"{x}|0")

In [1086]:
result_df['ICPE_A'] = result_df['ICPE_A'].str.split('|')
result_df = result_df.explode('ICPE_A')
result_df = result_df[result_df['ICPE_A'] != "nan"]

In [1087]:
#On met des NaN là où il ne doit pas y avoir de valeurs
result_df.loc[((result_df['ICPE_A'] != "0") & (result_df["ICPE_A"] != "nan")), "ICPE_E"] = np.nan
result_df.loc[((result_df['ICPE_A'] != "0") & (result_df["ICPE_A"] != "nan")), "ICPE_D"] = np.nan


In [1088]:
result_df['ICPE_E'] = result_df['ICPE_E'].str.split('|')
result_df = result_df.explode('ICPE_E')

In [1089]:
result_df = result_df[result_df['ICPE_E'] != "nan"]


In [1090]:
result_df.loc[((result_df['ICPE_E'] != "0") & (result_df["ICPE_E"] != "nan") & (result_df['ICPE_E'].notna() )), "ICPE_D"] = np.nan


In [1091]:
result_df['ICPE_D'] = result_df['ICPE_D'].str.split('|')
result_df = result_df.explode('ICPE_D')

In [1092]:
result_df = result_df[result_df['ICPE_D'] != "nan"]

In [1093]:
result_df.loc[(result_df['ICPE_A'] == "0"), "ICPE_A"] = np.nan
result_df.loc[(result_df['ICPE_E'] == "0"), "ICPE_E"] = np.nan


In [1094]:
result2 = result_df[result_df.duplicated(subset=colonnes_a_comparer, keep=False)]
result2.to_csv('result2_inter2.csv', index=False)
result2.loc[result2[['ICPE_A', 'ICPE_E', 'ICPE_D']].isna().all(axis=1), 'code_aiot'] = 0
result2[result2[['ICPE_A', 'ICPE_E', 'ICPE_D']].isna().all(axis=1)]


C:\Users\maxim\AppData\Local\Temp\ipykernel_12280\10071217.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  result2.loc[result2[['ICPE_A', 'ICPE_E', 'ICPE_D']].isna().all(axis=1), 'code_aiot'] = 0


,code_aiot,x,y,nom_ets,adresse,cd_insee,cd_postal,commune,code_naf,lib_naf,num_siret,cd_regime,lib_regime,seveso,lib_seveso,ICPE_A,ICPE_E,ICPE_D,date_modification,derniere_inspection
4,0,768626,6388530,EOLIEN - TOTAL QUADRAN CE La Luçoise,Forêt de Chaniaux - Lou Chambon,48086,48250,Luc,NaN,NaN,81519422000027,A,Autorisation,3,Non Seveso,NaN,NaN,NaN,2026/02/20 00:00:00,NaN
5,0,839475,6374347,COLLECTES VALORISATION ENERGIE DECHETS - COVED,AV DES EOLIENNES LE RAZAS,26169,26780,Malataverne,38,"Collecte, traitement et élimination des déchet...",34340353103542,A,Autorisation,3,Non Seveso,NaN,NaN,NaN,2026/03/18 00:00:00,2026/01/26 00:00:00
6,0,940539,6842661,NOVAWOOD,chemin du Vaquené,54300,54410,Laneuveville-devant-Nancy,NaN,NaN,82181865500010,A,Autorisation,3,Non Seveso,NaN,NaN,NaN,2026/06/17 00:00:00,2025/08/28 00:00:00
9,0,802493,6910500,Futures Energies du Mont Heudelan 2,Lieu-dit l'Arbre Malet - Les Blanches Voies et...,51487,51490,Saint-Hilaire-le-Petit,35,"Production et distribution d'électricité, de g...",80793671100058,A,Autorisation,3,Non Seveso,NaN,NaN,NaN,2026/04/25 00:00:00,2024/11/26 00:00:00
10,0,788739,6958278,FERME EOLIENNE LA HOTTE,1 place Jean Mermoz,08366,08220,Rocquigny,35,"Production et distribution d'électricité, de g...",80366621300019,A,Autorisation,3,Non Seveso,NaN,NaN,NaN,2026/04/25 00:00:00,2025/10/27 00:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
134152,0,698549,6810096,SAM MONTEREAU,36 RUE DE LA GRANDE HAIE - ZONE INDUSTRIELLE,77305,77130,Montereau-Fault-Yonne,24,Métallurgie,78497572400013,E,Enregistrement,3,Non Seveso,NaN,NaN,NaN,2026/05/01 00:00:00,2021/10/13 00:00:00
134240,0,656779,6602301,SAS BIOGAZ BURGMAYER,Lieu-dit Port Arthur,03225,03370,Saint-Désiré,NaN,NaN,85262329700011,E,Enregistrement,3,Non Seveso,NaN,NaN,NaN,2026/02/03 00:00:00,NaN
134385,0,700858,7036976,RECYCÂBLES,"1, rue de Malfidano",62624,62950,Noyelles-Godault,NaN,NaN,50174275300046,A,Autorisation,3,Non Seveso,NaN,NaN,NaN,2026/03/03 00:00:00,NaN
134805,0,862095,6514550,AEROPORTS DE LYON S.A. - TRISTAR,Aéroport de Lyon - Saint Exupéry,69299,69124,Colombier-Saugnieu,NaN,NaN,49342513600022,A,Autorisation,3,Non Seveso,NaN,NaN,NaN,2026/06/09 00:00:00,NaN


In [1097]:
print(result2[result2['code_aiot']==0].index)

Index([     4,      5,      6,      9,     10,     14,     24,     25,     26,
           27,
       ...
       133584, 133661, 133769, 133795, 133981, 134152, 134240, 134385, 134805,
       134848],
      dtype='int64', length=20733)


In [1098]:
result2.tail()

,code_aiot,x,y,nom_ets,adresse,cd_insee,cd_postal,commune,code_naf,lib_naf,num_siret,cd_regime,lib_regime,seveso,lib_seveso,ICPE_A,ICPE_E,ICPE_D,date_modification,derniere_inspection
134776,0100315431,736945,6636565,NIVERGRES,176 avenue de Verdun,58055,58300,Decize,NaN,NaN,NaN,A,Autorisation,3,Non Seveso,NaN,NaN,2910,2026/06/02 00:00:00,NaN
134805,0100316184,862095,6514550,AEROPORTS DE LYON S.A. - TRISTAR,Aéroport de Lyon - Saint Exupéry,69299,69124,Colombier-Saugnieu,NaN,NaN,49342513600022,A,Autorisation,3,Non Seveso,2712,NaN,NaN,2026/06/09 00:00:00,NaN
134805,0,862095,6514550,AEROPORTS DE LYON S.A. - TRISTAR,Aéroport de Lyon - Saint Exupéry,69299,69124,Colombier-Saugnieu,NaN,NaN,49342513600022,A,Autorisation,3,Non Seveso,NaN,NaN,NaN,2026/06/09 00:00:00,NaN
134848,0100317341,566051,6478435,SCEA GUILLOUX,SARRETTE,19025,19230,Beyssenac,01,"Culture et production animale, chasse et servi...",84831287200015,E,Enregistrement,3,Non Seveso,NaN,2102,NaN,2026/06/20 00:00:00,NaN
134848,0,566051,6478435,SCEA GUILLOUX,SARRETTE,19025,19230,Beyssenac,01,"Culture et production animale, chasse et servi...",84831287200015,E,Enregistrement,3,Non Seveso,NaN,NaN,NaN,2026/06/20 00:00:00,NaN


In [1096]:
result2 = result2.drop((result2[result2['code_aiot']==0]))
result2.tail(15)

KeyError: "['code_aiot' 'x' 'y' 'nom_ets' 'adresse' 'cd_insee' 'cd_postal' 'commune'\n 'code_naf' 'lib_naf' 'num_siret' 'cd_regime' 'lib_regime' 'seveso'\n 'lib_seveso' 'ICPE_A' 'ICPE_E' 'ICPE_D' 'date_modification'\n 'derniere_inspection'] not found in axis"

In [857]:
colonnes_a_comparer = result_df.columns.difference(['ICPE_A', 'ICPE_E', 'ICPE_D'])
mask_doublons = result_df.duplicated(subset=colonnes_a_comparer, keep=False)
mask_nan = result_df[['ICPE_A', 'ICPE_E', 'ICPE_D']].isna().all(axis=1)
result_df = result_df.drop(result_df[mask_doublons & mask_nan].index)
result_df.to_csv('result_inter.csv', index=False)

In [827]:
result_df = result_df.drop (result_df[(result_df[['ICPE_A', 'ICPE_E', 'ICPE_D']].isna().all(axis=1)) & (result_df.duplicated(subset=['code_aiot'], keep=False))].index)
result_df.to_csv('result_inter.csv', index=False)

### On ajoute au fichier les descriptions des codes ICPE correspondantes

In [673]:
#On ne garde que les codes ICPE complets et non pas les grandes catégories
nomenclature_df = nomenclature_df[nomenclature_df['Service'].notna()]

In [674]:
nomenclature_df.head(10)

,Code,Description,Service
2,1185,Gaz à effet de serre fluorés,SSEEC-BPC
5,1312,Mise en œuvre de produits explosifs à des fins...,SRT-BRIEC
8,1413,Installations de remplissage de réservoirs de ...,SRT-BRIEC
9,1414,Installations de remplissage ou de distributio...,SRT-BRIEC
10,1416,Station services (hydrogène),SRT-BRIEC
12,1421,Installation de remplissage d'aérosols inflamm...,SRT-BRIEC
14,1434,Installations de remplissage ou de distributio...,SRT-BRIEC
15,1435,Stations-service,SRT-BRIEC
16,1436,Liquides de point éclair compris entre 60°C et...,SRT-BRIEC
18,1450,Solides inflammables,SRT-BRIEC


In [574]:
# On convertit les types pour la concaténation
print(result_df.shape)
nomenclature_df['Code'].astype(int)
result_df['ICPE_A'].astype('Int64')
result.shape

(178088, 20)


(178088, 21)

In [575]:
result_df['ICPE'] = result_df['ICPE_A'].fillna('') + result_df['ICPE_E'].fillna('') +  result_df['ICPE_D'].fillna('')

In [576]:
# On merge
result = pd.merge(result_df, nomenclature_df, left_on='ICPE', right_on="Code", how="left")
result.shape

(178088, 24)

In [577]:
result=result.drop(columns=['ICPE', 'Code', 'Service'])
result.head(35)

,code_aiot,x,y,nom_ets,adresse,cd_insee,cd_postal,commune,code_naf,lib_naf,...,cd_regime,lib_regime,seveso,lib_seveso,ICPE_A,ICPE_E,ICPE_D,date_modification,derniere_inspection,Description
0,0000000002,432849,6444403,PITCH IMMO SNC : HEXAHUB : hangars logistiques...,Secteur de la Marquette,33183,33240,Gauriaguet,41,Construction de bâtiments,...,A,Autorisation,3,Non Seveso,1450,NaN,NaN,2026/05/06 00:00:00,NaN,Solides inflammables
1,0000000002,432849,6444403,PITCH IMMO SNC : HEXAHUB : hangars logistiques...,Secteur de la Marquette,33183,33240,Gauriaguet,41,Construction de bâtiments,...,A,Autorisation,3,Non Seveso,1510,NaN,NaN,2026/05/06 00:00:00,NaN,"Stockage de matières, produits ou substances c..."
2,0000000002,432849,6444403,PITCH IMMO SNC : HEXAHUB : hangars logistiques...,Secteur de la Marquette,33183,33240,Gauriaguet,41,Construction de bâtiments,...,A,Autorisation,3,Non Seveso,1630,NaN,NaN,2026/05/06 00:00:00,NaN,Emploi ou stockage de lessives de soude ou de ...
3,0000000002,432849,6444403,PITCH IMMO SNC : HEXAHUB : hangars logistiques...,Secteur de la Marquette,33183,33240,Gauriaguet,41,Construction de bâtiments,...,A,Autorisation,3,Non Seveso,4801,NaN,NaN,2026/05/06 00:00:00,NaN,"Houille, coke,..."
4,0000000002,432849,6444403,PITCH IMMO SNC : HEXAHUB : hangars logistiques...,Secteur de la Marquette,33183,33240,Gauriaguet,41,Construction de bâtiments,...,A,Autorisation,3,Non Seveso,NaN,NaN,1436,2026/05/06 00:00:00,NaN,Liquides de point éclair compris entre 60°C et...
5,0000000002,432849,6444403,PITCH IMMO SNC : HEXAHUB : hangars logistiques...,Secteur de la Marquette,33183,33240,Gauriaguet,41,Construction de bâtiments,...,A,Autorisation,3,Non Seveso,NaN,NaN,2171,2026/05/06 00:00:00,NaN,"Dépôts de fumiers, engrais et supports de culture"
6,0000000002,432849,6444403,PITCH IMMO SNC : HEXAHUB : hangars logistiques...,Secteur de la Marquette,33183,33240,Gauriaguet,41,Construction de bâtiments,...,A,Autorisation,3,Non Seveso,NaN,NaN,2910,2026/05/06 00:00:00,NaN,Installation de combustion
7,0000000002,432849,6444403,PITCH IMMO SNC : HEXAHUB : hangars logistiques...,Secteur de la Marquette,33183,33240,Gauriaguet,41,Construction de bâtiments,...,A,Autorisation,3,Non Seveso,NaN,NaN,2925,2026/05/06 00:00:00,NaN,Charge d'accumulateurs
8,0000000002,432849,6444403,PITCH IMMO SNC : HEXAHUB : hangars logistiques...,Secteur de la Marquette,33183,33240,Gauriaguet,41,Construction de bâtiments,...,A,Autorisation,3,Non Seveso,NaN,NaN,4110,2026/05/06 00:00:00,NaN,Toxicité aiguë catégorie 1
9,0000000002,432849,6444403,PITCH IMMO SNC : HEXAHUB : hangars logistiques...,Secteur de la Marquette,33183,33240,Gauriaguet,41,Construction de bâtiments,...,A,Autorisation,3,Non Seveso,NaN,NaN,4120,2026/05/06 00:00:00,NaN,Toxicité aiguë catégorie 2


In [578]:
# On change l'ordre des colonnes pour plus de lisibilité
result= result[['code_aiot', 'nom_ets', 'x', 'y', 'adresse', 'cd_insee', 'cd_postal',
       'commune', 'code_naf', 'lib_naf', 'num_siret', 'cd_regime',
       'lib_regime', 'seveso', 'lib_seveso', 'Description', 'ICPE_A', 'ICPE_E', 'ICPE_D',
       'date_modification', 'derniere_inspection']]
result= result.rename({'Description':'Description du code ICPE'}, axis = 1)

In [579]:
# On récupère un excel trié
result.to_csv('result_trie.csv', index=False)